# 00 — Build the RAG dataset (retrieval corpus)

Creates **`data/processed/rag_corpus.csv`** — the pool of labelled *precedent* loans the two RAG notebooks retrieve from. It is the **full large dataset with every evaluation batch removed**, so no test loan can ever be retrieved.

**Source pool (auto):**
1. The full 2012–2014 LendingClub frame from the raw `data/raw/accepted_2007_to_2018Q4.csv.gz` (via `sample_generation._build_frame`). The real, large corpus.
2. **Dev fallback** — if the raw file is absent: the committed `tuning_sample ∪ robustness_batch` (200 rows, same 35-col schema, disjoint from `test_batch`). Lets everything run before the raw file is added.

Either way the **1000-row `test_batch.csv`** (and, for the full frame, the tuning/robustness batches) are removed, and zero overlap with the test set is asserted.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import rag_utils as R

In [2]:
# Build (force=True regenerates). Drop the raw .csv.gz into data/raw/ and rerun
# with force=True to swap the dev fallback for the full corpus.
corpus = R.build_rag_corpus(force=True)
test   = pd.read_csv(R.TEST_BATCH_PATH)
R.assert_no_leakage(corpus, test)
print(f'RAG corpus rows : {len(corpus)}')
print(f'Test batch rows : {len(test)}')
print('Leakage check   : PASSED (corpus ∩ test = ∅)')

Built RAG dataset from dev_fallback: kept 200/200 rows (0 removed as eval/leakage/dupes).
  Class balance — Charged Off: 24  Fully Paid: 176
  Saved -> C:\Users\Jad Zoghaib\OneDrive\Desktop\Sabadell_Capstone\data\processed\rag_corpus.csv
RAG corpus rows : 200
Test batch rows : 1000
Leakage check   : PASSED (corpus ∩ test = ∅)


In [3]:
# Quick profile of the corpus
co = int((corpus['loan_status'] == 0).sum())
print(f'Charged Off: {co}  |  Fully Paid: {len(corpus) - co}')
corpus.head()

Charged Off: 24  |  Fully Paid: 176


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,mths_since_last_delinq,acc_open_past_24mths,credit_history_years,has_past_delinq,total_pymnt,total_rec_prncp,total_rec_int,recoveries,collection_recovery_fee,funded_amnt
0,18950.0,36 months,13.11,639.51,B,B4,10.0,RENT,60500.0,Source Verified,...,999.0,2.0,13.582478,0,NaN,NaN,NaN,NaN,NaN,NaN
1,8000.0,36 months,14.09,273.78,B,B5,7.0,MORTGAGE,40000.0,Verified,...,46.0,5.0,28.000000,1,NaN,NaN,NaN,NaN,NaN,NaN
2,15000.0,36 months,9.67,481.69,B,B1,7.0,OWN,100000.0,Not Verified,...,24.0,2.0,13.248460,1,NaN,NaN,NaN,NaN,NaN,NaN
3,24000.0,36 months,6.62,736.89,A,A2,10.0,MORTGAGE,120000.0,Not Verified,...,60.0,3.0,18.915811,1,NaN,NaN,NaN,NaN,NaN,NaN
4,8825.0,36 months,14.09,302.01,B,B5,6.0,MORTGAGE,93000.0,Verified,...,77.0,3.0,20.503765,1,NaN,NaN,NaN,NaN,NaN,NaN
